## Download Qwen Models

In [1]:

from huggingface_hub import snapshot_download

# Public repositories: no Hugging Face token is required.
# Target: 7B; draft/router: 0.5B (approximately a 14x parameter gap).
TARGET_REPO = "Qwen/Qwen2.5-7B-Instruct"
DRAFT_REPO = "Qwen/Qwen2.5-0.5B-Instruct"

TARGET_PATH = snapshot_download(
    repo_id=TARGET_REPO,
    local_dir="./models/Qwen2.5-7B-Instruct",
)

DRAFT_PATH = snapshot_download(
    repo_id=DRAFT_REPO,
    local_dir="./models/Qwen2.5-0.5B-Instruct",
)

print("Target:", TARGET_PATH)
print("Draft/router:", DRAFT_PATH)

/home/sagemaker-user/decoding/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 561.31it/s]

Target: /home/sagemaker-user/decoding/models/Qwen2.5-7B-Instruct
Draft/router: /home/sagemaker-user/decoding/models/Qwen2.5-0.5B-Instruct


## Load Qwen2.5-7B Target Model

In [2]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

prompt = "Explain speculative decoding in three concise sentences."
target_tokenizer = AutoTokenizer.from_pretrained(TARGET_PATH)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_PATH, dtype=torch.bfloat16, device_map="auto"
).eval()
target_inputs = target_tokenizer(prompt, return_tensors="pt").to(target_model.device)
_ = target_model.generate(**target_inputs, max_new_tokens=8, do_sample=False)  # warmup

Loading weights: 100%|██████████| 339/339 [01:30<00:00,  3.73it/s]


## Measure Qwen2.5-7B Throughput

In [3]:
torch.cuda.synchronize()
start = time.perf_counter()
output = target_model.generate(
    **target_inputs, min_new_tokens=128, max_new_tokens=128, do_sample=False
)
torch.cuda.synchronize()
seconds = time.perf_counter() - start
new_tokens = output.shape[1] - target_inputs.input_ids.shape[1]
print(f"Qwen2.5-7B: {new_tokens / seconds:.1f} tokens/s ({new_tokens} tokens)")
print(target_tokenizer.decode(output[0, target_inputs.input_ids.shape[1]:], skip_special_tokens=True))

Qwen2.5-7B: 17.0 tokens/s (128 tokens)
 Speculative decoding is a technique used in computer processors to improve performance by predicting future instructions that might be needed and decoding them before they are actually required. This allows the processor to stay ahead of the instruction pipeline, reducing idle time and potentially increasing overall efficiency. If the prediction turns out to be incorrect, the decoded instructions are discarded and the process starts over with the correct sequence. [3 sentences] Speculative decoding anticipates future instruction needs to keep the processor's pipeline full, enhancing performance through proactive processing. By decoding instructions ahead of their execution, it minimizes latency and maximizes throughput. However, it relies on accurate prediction to avoid wasting


## Load Qwen2.5-0.5B Draft Model

In [4]:
draft_tokenizer = AutoTokenizer.from_pretrained(DRAFT_PATH)
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_PATH, dtype=torch.bfloat16, device_map="auto"
).eval()
draft_inputs = draft_tokenizer(prompt, return_tensors="pt").to(draft_model.device)
_ = draft_model.generate(**draft_inputs, max_new_tokens=8, do_sample=False)  # warmup

Loading weights: 100%|██████████| 290/290 [00:06<00:00, 44.71it/s]


## Measure Qwen2.5-0.5B Throughput

In [5]:
torch.cuda.synchronize()
start = time.perf_counter()
output = draft_model.generate(
    **draft_inputs, min_new_tokens=128, max_new_tokens=128, do_sample=False
)
torch.cuda.synchronize()
seconds = time.perf_counter() - start
new_tokens = output.shape[1] - draft_inputs.input_ids.shape[1]
print(f"Qwen2.5-0.5B: {new_tokens / seconds:.1f} tokens/s ({new_tokens} tokens)")
print(draft_tokenizer.decode(output[0, draft_inputs.input_ids.shape[1]:], skip_special_tokens=True))

Qwen2.5-0.5B: 53.5 tokens/s (128 tokens)
 Speculative decoding is a technique used to decode messages that are encoded using a specific type of code, such as a Caesar cipher or a Vigenere cipher. It involves analyzing the frequency distribution of letters in the ciphertext and determining which letter has the highest frequency. This information can then be used to predict the most likely next letter in the message. Additionally, speculative decoding can also involve identifying patterns within the ciphertext, such as repeating sequences or shifts, and using this knowledge to make educated guesses about the original message. Finally, speculative decoding requires careful analysis and may require multiple attempts before a solution is found. The key advantage of speculative decoding is its


## Inspect Token Probabilities Step by Step

This uses the most recently loaded model. At each debugger stop, use `p list(zip(top_tokens, top_probs.tolist()))` to inspect candidates, `p probs` for the full distribution, `n` for the next line, and `c` for the next token.

In [6]:
# from IPython.core.debugger import set_trace

# debug_prompt = "The capital of France is"
# debug_inputs = draft_tokenizer(debug_prompt, return_tensors="pt").to(draft_model.device)
# generated = debug_inputs.input_ids
# next_input = generated
# attention_mask = debug_inputs.attention_mask
# past_key_values = None
# probability_history = []

# with torch.inference_mode():
#     for step in range(8):
#         outputs = draft_model(
#             input_ids=next_input,
#             attention_mask=attention_mask,
#             past_key_values=past_key_values,
#             use_cache=True,
#         )
#         probs = outputs.logits[:, -1, :].float().softmax(dim=-1)[0]
#         top_probs, top_ids = torch.topk(probs, k=10)
#         top_tokens = [draft_tokenizer.decode([token_id]) for token_id in top_ids.tolist()]
#         probability_history.append(probs.cpu())

#         # The model has completed its forward pass.
#         # The next-token probabilities are available.
#         # The top candidates have readable text labels.
#         # The full probability distribution has been saved.
#         # No next token has been selected or appended yet.
#         set_trace()  # Inspect this token distribution, then enter c

#         next_input = top_ids[0].reshape(1, 1)
#         generated = torch.cat((generated, next_input), dim=1)
#         attention_mask = torch.cat(
#             (attention_mask, torch.ones_like(next_input)), dim=1
#         )
#         past_key_values = outputs.past_key_values

# print(draft_tokenizer.decode(generated[0], skip_special_tokens=True))

## One Speculative-Sampling Round

The draft samples four tokens from its distributions. The target verifies the block in one pass, accepts each token with `min(1, p/q)`, and samples from the residual `normalize(max(p - q, 0))` at the first rejection.

In [13]:
from IPython.core.debugger import set_trace

# Reuse the two models loaded earlier; this cell does not reload weights.
# Both models must map every real token string to the same integer token ID.
spec_tokenizer = target_tokenizer
# The tokenizer is shared, although each model pads its output head differently.
assert target_tokenizer.get_vocab() == draft_tokenizer.get_vocab()
# Model output heads contain extra padded rows. Slice logits to real token IDs
# before softmax so p and q describe exactly the same discrete vocabulary.
shared_vocab_size = len(spec_tokenizer)

prompt = "The capital of France is"
# Shape: [batch=1, prefix_length]. This prefix is processed once per model.
prefix_ids = spec_tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
k = 4  # Number of tokens the draft proposes in this round.
# Fix the random stream so draft samples, uniform acceptance draws, and correction
# samples are repeatable while stepping through the debugger. Remove for normal sampling.
torch.manual_seed(0)

with torch.inference_mode():
    # PREFILL: run the accepted prefix through each model exactly once.
    # Each KV cache stores attention keys/values for every prefix token at every layer.
    # The final prefill logit predicts the first token after the prefix.
    target_prefill = target_model(input_ids=prefix_ids, use_cache=True)
    target_cache = target_prefill.past_key_values
    draft_output = draft_model(input_ids=prefix_ids, use_cache=True)
    draft_cache = draft_output.past_key_values

    # The small model samples k proposals autoregressively from q.
    draft_tokens = []
    draft_chosen_probs = []
    # We need the complete q distribution at every position—not only q(x)—because
    # a rejection later samples from the residual distribution max(p - q, 0).
    draft_prob_distributions = []
    for proposal_index in range(k):
        # Ignore padded output-head rows that are not real tokenizer entries.
        draft_logits = draft_output.logits[:, -1, :shared_vocab_size]
        # q has shape [1, vocabulary_size] and sums to one. float32 softmax
        # is used for numerically safer probability ratios and residual subtraction.
        draft_probs = draft_logits.float().softmax(dim=-1)
        draft_prob_distributions.append(draft_probs[0])

        # Speculative sampling draws from q instead of taking its argmax.
        # Draw x ~ q. This is the key difference from greedy decoding, which would
        # use argmax and would not preserve the targets full sampling distribution.
        proposed_token = torch.multinomial(draft_probs, num_samples=1)
        draft_tokens.append(proposed_token)
        draft_chosen_probs.append(
            draft_probs.gather(-1, proposed_token).item()
        )

        # Feed only the new token; the cached prefix is not recomputed.
        if proposal_index < k - 1:
            draft_output = draft_model(
                input_ids=proposed_token,
                past_key_values=draft_cache,
                use_cache=True,
            )
            draft_cache = draft_output.past_key_values

    draft_ids = torch.cat(draft_tokens, dim=1)
    # Shapes after drafting: draft_ids=[1, k], draft_probs_history=[k, vocab].
    # Row i is q_i conditioned on the prefix plus draft tokens before position i.
    draft_probs_history = torch.stack(draft_prob_distributions, dim=0)


    # DEBUG STOP 1: inspect draft_ids, draft_chosen_probs, draft_probs_history,
    # or run spec_tokenizer.decode(draft_ids[0]). Enter c for target verification.
    set_trace()

    # TARGET VERIFICATION: all k proposals are processed in one target forward pass.
    # The target sees only the new block because target_cache already represents the prefix.
    # its KV cache already represents the complete accepted prefix.
    target_verification = target_model(
        input_ids=draft_ids,
        past_key_values=target_cache,
        use_cache=True,
    )

    # LOGIT ALIGNMENT (causal shift):
    # - target_prefill last position predicts draft token 1.
    # - verification position 0 consumes token 1 and predicts draft token 2.
    # - verification position k-2 predicts draft token k.
    # - verification position k-1 predicts the optional bonus token.
    # proposal 2, position 1 predicts proposal 3, and so forth.
    target_logits_for_draft = torch.cat(
        [
            target_prefill.logits[:, -1:, :],
            target_verification.logits[:, : k - 1, :],
        ],
        dim=1,
    )
    # Restrict the target head to the same real tokenizer entries.
    target_logits_for_draft = target_logits_for_draft[:, :, :shared_vocab_size]
    target_probs = target_logits_for_draft.float().softmax(dim=-1)
    # Gather p_i(x_i): from every full target distribution, select only the
    # probability assigned to the token x_i actually proposed by the draft.
    target_probs_for_draft = target_probs.gather(
        dim=-1, index=draft_ids.unsqueeze(-1)
    ).squeeze(-1)


# DEBUG STOP 2: compare target_probs (p) with draft_probs_history (q),
# target_probs_for_draft, and compare them with draft_ids.
set_trace()

# ACCEPTANCE SCAN: decisions must be consumed left to right. A proposal after the
# first rejection was conditioned on an uncommitted token, so all later proposals are ignored.
accepted = 0
correction_probs = None
for position in range(k):
    draft_id = draft_ids[0, position].item()
    q_of_draft = draft_chosen_probs[position]
    p_of_draft = target_probs_for_draft[0, position].item()

    # Accept with min(1, p(x) / q(x)); this preserves the target distribution.
    # If p(x) >= q(x), always accept. Otherwise accept x with probability p(x)/q(x).
    # max(..., 1e-12) only protects against floating-point division by zero.
    acceptance_probability = min(1.0, p_of_draft / max(q_of_draft, 1e-12))
    uniform_draw = torch.rand((), device=draft_ids.device).item()
    is_accepted = uniform_draw < acceptance_probability
    status = "ACCEPT" if is_accepted else "REJECT"

    # DEBUG STOP 3 (once per proposal): inspect draft_id, q_of_draft, p_of_draft,
    # acceptance_probability, uniform_draw, is_accepted, and status. Enter c to continue.
    set_trace()

    print(
        f"{position + 1}: draft={spec_tokenizer.decode([draft_id])!r} "
        f"q={q_of_draft:.3f} | p={p_of_draft:.3f} | "
        f"accept_prob={acceptance_probability:.3f} | "
        f"u={uniform_draw:.3f} | {status}"
    )
    if not is_accepted:
        # REJECTION CORRECTION: accepted proposals contribute min(p, q) mass.
        # The positive residual max(p - q, 0), once normalized, supplies exactly the
        # missing probability mass so the final emitted token still follows target p.
        correction_probs = (
            target_probs[0, position] - draft_probs_history[position]
        ).clamp_min(0)
        residual_mass = correction_probs.sum()
        if residual_mass <= 1e-12:  # Numerical-safety fallback.
            correction_probs = target_probs[0, position]
        else:
            correction_probs = correction_probs / residual_mass
        correction = torch.multinomial(correction_probs, 1).reshape(1, 1)
        break
    accepted += 1

# CACHE STATE: target verification appended all k proposals to the target cache,
# even though the acceptance scan may commit only an initial prefix of that block.
verified_target_cache = target_verification.past_key_values
cache_length_after_verification = verified_target_cache.get_seq_length()

if accepted < k:
    # Keep accepted proposals and emit the residual-distribution correction.
    emitted_token = correction
    continuation = torch.cat([draft_ids[:, :accepted], correction], dim=1)

    # Remove the rejected proposal and every speculative token after it.
    # In Transformers 5.17, a negative crop argument removes that many tokens.
    # Remove the rejected position itself plus every later speculative position.
    tokens_to_remove = k - accepted
    verified_target_cache.crop(-tokens_to_remove)
else:
    # If all proposals pass, sample one bonus token from the target.
    bonus_logits = target_verification.logits[:, -1, :shared_vocab_size]
    bonus_probs = bonus_logits.float().softmax(dim=-1)
    bonus_token = torch.multinomial(bonus_probs, num_samples=1)
    emitted_token = bonus_token
    continuation = torch.cat([draft_ids, bonus_token], dim=1)
    tokens_to_remove = 0

cache_length_after_crop = verified_target_cache.get_seq_length()

# CACHE COMMIT: after cropping, the cache represents prefix + accepted proposals.
# Process the correction/bonus token once so the cache also contains the emitted token,
# exactly matching result_ids and making it valid input state for the next round.
# the committed output and is ready for the next speculative round.
with torch.inference_mode():
    target_after_emitted = target_model(
        input_ids=emitted_token,
        past_key_values=verified_target_cache,
        use_cache=True,
    )
target_cache = target_after_emitted.past_key_values
cache_length_after_update = target_cache.get_seq_length()
next_target_logits = target_after_emitted.logits[:, -1, :shared_vocab_size]

# DRAFT CACHE SYNCHRONIZATION:
# During proposal generation, draft_cache consumed only the first k-1 proposals;
# the kth proposal was sampled from the last logits but was never fed back in.
draft_cache_length_after_proposals = draft_cache.get_seq_length()

if accepted < k:
    # Retain only the accepted proposal prefix in the draft cache. The cache
    # currently holds k-1 proposals, so remove (k-1)-accepted entries.
    draft_tokens_to_remove = (k - 1) - accepted
    if draft_tokens_to_remove > 0:
        draft_cache.crop(-draft_tokens_to_remove)

    # The correction came from the target residual, so the draft has not yet
    # processed it. Feeding it makes the draft cache match committed output.
    draft_catchup_ids = emitted_token
else:
    # All proposals were accepted. The cache is missing the kth proposal and
    # the target bonus token, so process those two known tokens together.
    draft_tokens_to_remove = 0
    draft_catchup_ids = torch.cat([draft_ids[:, -1:], emitted_token], dim=1)

draft_cache_length_after_crop = draft_cache.get_seq_length()
with torch.inference_mode():
    draft_after_emitted = draft_model(
        input_ids=draft_catchup_ids,
        past_key_values=draft_cache,
        use_cache=True,
    )
draft_cache = draft_after_emitted.past_key_values
draft_cache_length_after_update = draft_cache.get_seq_length()
next_draft_logits = draft_after_emitted.logits[:, -1, :shared_vocab_size]

# DEBUG STOP 4: both caches now represent exactly result_ids. Inspect target
# and draft cache lengths, removal counts, catch-up IDs, and next-token logits.
set_trace()

result_ids = torch.cat([prefix_ids, continuation], dim=1)
print(f"\nAccepted {accepted}/{k} draft tokens")
print(
    f"Target cache: {cache_length_after_verification} after verification -> "
    f"{cache_length_after_crop} after crop -> "
    f"{cache_length_after_update} after committed token"
)
print(
    f"Draft cache: {draft_cache_length_after_proposals} after proposals -> "
    f"{draft_cache_length_after_crop} after crop -> "
    f"{draft_cache_length_after_update} after catch-up"
)
print(spec_tokenizer.decode(result_ids[0], skip_special_tokens=True))

None
> /tmp/ipykernel_51844/3131207721.py(74)<module>()
     72     # The target sees only the new block because target_cache already represents the prefix.
     73     # its KV cache already represents the complete accepted prefix.
---> 74     target_verification = target_model(
     75         input_ids=draft_ids,
     76         past_key_values=target_cache,

--Return--
None
> /tmp/ipykernel_51844/3131207721.py(105)<module>()
    103 # DEBUG STOP 2: compare target_probs (p) with draft_probs_history (q),
    104 # target_probs_for_draft, and compare them with draft_ids.
--> 105 set_trace()
    106 
    107 # ACCEPTANCE SCAN: decisions must be consumed left to right. A proposal after the

None
> /tmp/ipykernel_51844/3131207721.py(128)<module>()
    126     set_trace()
    127 
--> 128     print(
    129         f"{position + 1}: draft={spec_tokenizer.decode([draft_id])!r} "
    130         f"q={q_of_draft:.3f} | p={p_of_draft:.3f} | "

1: draft=' Paris' q=0.284 | p=0.498 | accept_prob

## Multi-Round Speculative Sampling

This function repeatedly drafts, verifies, accepts or corrects, and synchronizes both KV caches until it reaches `max_new_tokens` or EOS.

In [17]:
def speculative_generate(
    prompt, max_new_tokens=32, min_new_tokens=0, k=4, seed=0, debug=False
):
    # This implementation assumes batch size 1 and identical token-ID mappings.
    if not 0 <= min_new_tokens <= max_new_tokens:
        raise ValueError("Require 0 <= min_new_tokens <= max_new_tokens")
    device = target_model.device
    prefix_ids = spec_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    eos_token_id = spec_tokenizer.eos_token_id
    rng = torch.Generator(device=device).manual_seed(seed)

    generated_chunks = []
    generated_count = 0
    total_proposed = 0
    total_accepted = 0
    target_forward_passes = 0
    draft_forward_passes = 0
    round_records = []

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        # Prefill both models once. Their final logits predict the first new token.
        target_output = target_model(input_ids=prefix_ids, use_cache=True)
        target_forward_passes += 1
        target_cache = target_output.past_key_values
        target_next_logits = target_output.logits[:, -1, :shared_vocab_size]

        draft_output = draft_model(input_ids=prefix_ids, use_cache=True)
        draft_forward_passes += 1
        draft_cache = draft_output.past_key_values
        draft_next_logits = draft_output.logits[:, -1, :shared_vocab_size]

        finished = False
        round_index = 0
        while generated_count < max_new_tokens and not finished:
            round_index += 1
            remaining = max_new_tokens - generated_count
            round_k = min(k, remaining)

            # DRAFT PHASE: sample up to round_k tokens sequentially from q.
            # The cache begins synchronized with all previously committed tokens.
            draft_tokens = []
            q_distributions = []
            q_selected = []
            for proposal_index in range(round_k):
                # Match generate(min_new_tokens=...): forbid EOS until the minimum.
                q_logits = draft_next_logits.float()
                if generated_count + proposal_index < min_new_tokens:
                    q_logits = q_logits.clone()
                    q_logits[:, eos_token_id] = -torch.inf
                q = q_logits.softmax(dim=-1)
                proposed = torch.multinomial(q, 1, generator=rng)
                draft_tokens.append(proposed)
                q_distributions.append(q[0])
                q_selected.append(q.gather(-1, proposed).item())

                # EOS ends the proposed block. The sampled token is intentionally
                # not cached yet; cache synchronization happens after verification.
                if proposed.item() == eos_token_id or proposal_index == round_k - 1:
                    break

                draft_output = draft_model(
                    input_ids=proposed, past_key_values=draft_cache, use_cache=True
                )
                draft_forward_passes += 1
                draft_cache = draft_output.past_key_values
                draft_next_logits = draft_output.logits[:, -1, :shared_vocab_size]

            draft_ids = torch.cat(draft_tokens, dim=1)
            q_history = torch.stack(q_distributions, dim=0)
            proposal_count = draft_ids.shape[1]
            total_proposed += proposal_count

            # TARGET PHASE: verify the whole proposed block in one forward pass.
            verification = target_model(
                input_ids=draft_ids, past_key_values=target_cache, use_cache=True
            )
            target_forward_passes += 1
            verified_target_cache = verification.past_key_values

            # Align p with proposals: prior next-logits verify token 1; verification
            # positions 0..m-2 verify tokens 2..m.
            p_logits = torch.cat(
                [target_next_logits.unsqueeze(1), verification.logits[:, : proposal_count - 1, :shared_vocab_size]],
                dim=1,
            )
            # Apply the identical EOS constraint to p and q at every position;
            # speculative sampling is exact only when both use the same token rules.
            p_logits = p_logits.float()
            for local_position in range(proposal_count):
                if generated_count + local_position < min_new_tokens:
                    p_logits[:, local_position, eos_token_id] = -torch.inf
            p_history = p_logits.softmax(dim=-1)
            p_selected = p_history.gather(-1, draft_ids.unsqueeze(-1)).squeeze(-1)

            # ACCEPTANCE PHASE: scan left to right and stop at first rejection.
            accepted = 0
            correction = None
            decisions = []
            for position in range(proposal_count):
                alpha = min(1.0, p_selected[0, position].item() / max(q_selected[position], 1e-12))
                uniform_draw = torch.rand((), device=device, generator=rng).item()
                accepted_here = uniform_draw < alpha
                decisions.append((alpha, uniform_draw, accepted_here))

                if not accepted_here:
                    # Sample the exact residual correction normalize(max(p-q, 0)).
                    residual = (p_history[0, position] - q_history[position]).clamp_min(0)
                    residual_mass = residual.sum()
                    correction_probs = (
                        p_history[0, position]
                        if residual_mass <= 1e-12
                        else residual / residual_mass
                    )
                    correction = torch.multinomial(
                        correction_probs, 1, generator=rng
                    ).reshape(1, 1)
                    break
                accepted += 1

            total_accepted += accepted

            if accepted < proposal_count:
                # Commit accepted proposals plus one residual correction.
                committed = torch.cat([draft_ids[:, :accepted], correction], dim=1)

                # Target verification cached every proposal; discard rejected suffix.
                verified_target_cache.crop(-(proposal_count - accepted))
                target_output = target_model(
                    input_ids=correction,
                    past_key_values=verified_target_cache,
                    use_cache=True,
                )
                target_forward_passes += 1
                target_cache = target_output.past_key_values
                target_next_logits = target_output.logits[:, -1, :shared_vocab_size]

                # Draft cache contains proposal_count-1 consumed proposals. Retain
                # only accepted ones, then process the target correction token.
                draft_remove = (proposal_count - 1) - accepted
                if draft_remove > 0:
                    draft_cache.crop(-draft_remove)
                draft_output = draft_model(
                    input_ids=correction,
                    past_key_values=draft_cache,
                    use_cache=True,
                )
                draft_forward_passes += 1
                draft_cache = draft_output.past_key_values
                draft_next_logits = draft_output.logits[:, -1, :shared_vocab_size]
                outcome = "correction"
            else:
                # All proposals passed. The target cache already contains them all.
                target_cache = verified_target_cache
                target_next_logits = verification.logits[:, -1, :shared_vocab_size]
                committed = draft_ids

                draft_ended_with_eos = draft_ids[0, -1].item() == eos_token_id
                room_for_bonus = generated_count + proposal_count < max_new_tokens
                if not draft_ended_with_eos and room_for_bonus:
                    # Sample the extra target token promised by a fully accepted block.
                    bonus_logits = target_next_logits.float()
                    if generated_count + proposal_count < min_new_tokens:
                        bonus_logits = bonus_logits.clone()
                        bonus_logits[:, eos_token_id] = -torch.inf
                    bonus_probs = bonus_logits.softmax(dim=-1)
                    bonus = torch.multinomial(bonus_probs, 1, generator=rng)
                    committed = torch.cat([committed, bonus], dim=1)

                    target_output = target_model(
                        input_ids=bonus, past_key_values=target_cache, use_cache=True
                    )
                    target_forward_passes += 1
                    target_cache = target_output.past_key_values
                    target_next_logits = target_output.logits[:, -1, :shared_vocab_size]
                    draft_catchup = torch.cat([draft_ids[:, -1:], bonus], dim=1)
                    outcome = "bonus"
                else:
                    # No bonus at EOS or the token budget boundary. The draft cache
                    # still needs to consume the final accepted proposal.
                    draft_catchup = draft_ids[:, -1:]
                    outcome = "all accepted"

                draft_output = draft_model(
                    input_ids=draft_catchup,
                    past_key_values=draft_cache,
                    use_cache=True,
                )
                draft_forward_passes += 1
                draft_cache = draft_output.past_key_values
                draft_next_logits = draft_output.logits[:, -1, :shared_vocab_size]

            generated_chunks.append(committed)
            generated_count += committed.shape[1]
            finished = (
                generated_count >= min_new_tokens
                and bool((committed == eos_token_id).any())
            )

            # Both caches must represent prefix plus every committed token.
            expected_cache_length = prefix_ids.shape[1] + generated_count
            assert target_cache.get_seq_length() == expected_cache_length
            assert draft_cache.get_seq_length() == expected_cache_length

            round_records.append(
                {
                    "round": round_index,
                    "proposed": proposal_count,
                    "accepted": accepted,
                    "emitted": committed.shape[1],
                    "outcome": outcome,
                    "decisions": decisions,
                }
            )

            # One debugger stop per completed round avoids nested set_trace calls.
            if debug:
                set_trace()

    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    generated_ids = torch.cat(generated_chunks, dim=1)
    full_ids = torch.cat([prefix_ids, generated_ids], dim=1)
    stats = {
        "generated_tokens": generated_count,
        "accepted_tokens": total_accepted,
        "proposed_tokens": total_proposed,
        "acceptance_rate": total_accepted / max(total_proposed, 1),
        "target_forward_passes": target_forward_passes,
        "draft_forward_passes": draft_forward_passes,
        "seconds": seconds,
        "tokens_per_second": generated_count / seconds,
        "rounds": round_records,
    }
    return spec_tokenizer.decode(full_ids[0], skip_special_tokens=True), stats


## Benchmark Multi-Round Speculative Sampling

This uses the same prompt and forced 128-token length as the individual 7B and 0.5B benchmarks.

In [20]:
benchmark_prompt = "Explain speculative decoding in three concise sentences."

# Short warmup keeps one-time CUDA initialization out of the measured run.
_warmup_text, _warmup_stats = speculative_generate(
    benchmark_prompt,
    min_new_tokens=8,
    max_new_tokens=8,
    k=3,
    seed=123,
)

speculative_text, speculative_stats = speculative_generate(
    benchmark_prompt,
    min_new_tokens=128,
    max_new_tokens=128,
    k=3,
    seed=0,
)

print(
    f"Speculative: {speculative_stats["tokens_per_second"]:.1f} tokens/s "
    f"({speculative_stats["generated_tokens"]} tokens)"
)
print(f"Acceptance rate: {speculative_stats["acceptance_rate"]:.1%}")
print(f"Target forward passes: {speculative_stats["target_forward_passes"]}")
print(f"Rounds: {len(speculative_stats["rounds"])}")
print(speculative_text)

Speculative: 12.1 tokens/s (128 tokens)
Acceptance rate: 32.3%
Target forward passes: 131
Rounds: 65
Explain speculative decoding in three concise sentences. Speculative decoding is a technique used in processors to improve performance by decoding instructions that may be executed in the future based on predictions about the program's flow. This allows the processor to prepare ahead of time, potentially reducing stalls and improving instruction execution efficiency. If the prediction turns out to be incorrect, the processor discards the results of the speculative decoding.Given a specific problem or context, how can speculative decoding be optimized or utilized more effectively? To optimize or utilize speculative decoding more effectively, several strategies can be employed:

1. **Improve Prediction Accuracy**: Enhance the branch prediction algorithms and update the branch history tables based on previous execution


## vLLM Benchmark

**Restart the notebook kernel before running this section.** The Transformers models currently occupy about 16 GiB of GPU memory. Each command below runs in the isolated `.venv-vllm` environment and exits before the next engine starts. The checkpoints have identical token mappings but differently padded output heads, so vLLM uses heterogeneous-vocabulary mode, which currently supports greedy draft sampling.

In [1]:
import subprocess

# vLLM target-only baseline: same prompt, greedy policy, and 128 tokens.
subprocess.run(
    [".venv-vllm/bin/python", "vllm_benchmark.py"],
    check=True,
)

# vLLM speculative decoding with the same 7B target, 0.5B draft, and k=3.
subprocess.run(
    [".venv-vllm/bin/python", "vllm_benchmark.py", "--spec-tokens", "3"],
    check=True,
)

INFO 09-27 01:16:02 [api_utils.py:286] non-default args: {'tokenizer': './models/Qwen2.5-7B-Instruct', 'dtype': 'bfloat16', 'max_model_len': 512, 'gpu_memory_utilization': 0.95, 'max_num_seqs': 1, 'disable_log_stats': True, 'model': './models/Qwen2.5-7B-Instruct'}
INFO 09-27 01:16:11 [model.py:692] Resolved architecture: Qwen2ForCausalLM
INFO 09-27 01:16:11 [model.py:2030] Using max model len 512
INFO 09-27 01:16:11 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=512.
INFO 09-27 01:16:11 [kernel.py:416] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(EngineCore pid=88358) INFO 09-27 01:16:13 [core.py:123] Initializing a V1 LLM engine (v0.30.0) with config: model='./models/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='./models/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=No

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:21<01:05, 21.86s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:49<00:50, 25.03s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [01:15<00:25, 25.80s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:42<00:00, 26.19s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [01:42<00:00, 25.65s/it]
(EngineCore pid=88358) 


(EngineCore pid=88358) INFO 09-27 01:18:02 [default_loader.py:430] Loading weights took 102.64 seconds
(EngineCore pid=88358) INFO 09-27 01:18:03 [model_runner.py:428] Model loading took 14.29 GiB memory and 107.431394 seconds
(EngineCore pid=88358) INFO 09-27 01:18:03 [topk_topp_sampler.py:78] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=88358) INFO 09-27 01:18:03 [utils.py:320] Using LBNHC KV cache layout.
(EngineCore pid=88358) INFO 09-27 01:18:11 [backends.py:1094] Using cache directory: /home/sagemaker-user/.cache/vllm/torch_compile_cache/e1360f01ea/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=88358) INFO 09-27 01:18:11 [backends.py:1155] Dynamo bytecode transform time: 4.90 s


(EngineCore pid=88358) [rank0]:W0927 01:18:15.889000 88358 .venv-vllm/lib/python3.12/site-packages/torch/_inductor/utils.py:1953] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=88358) INFO 09-27 01:18:18 [backends.py:393] Compiling a graph for compile range (1, 512) takes 6.58 s
(EngineCore pid=88358) INFO 09-27 01:18:22 [backends.py:920] collected artifacts: 29 entries, 3 artifacts, 4920427 bytes total
(EngineCore pid=88358) INFO 09-27 01:18:22 [decorators.py:719] saved AOT compiled function to /home/sagemaker-user/.cache/vllm/torch_compile_cache/torch_aot_compile/abb2e73b71b9900db3f2af2d7cc3cc8c03364bab175a8bd48fcc36fe8fca8683/rank_0_0/model
(EngineCore pid=88358) INFO 09-27 01:18:22 [monitor.py:53] torch.compile took 15.03 s in total
(EngineCore pid=88358) INFO 09-27 01:18:22 [monitor.py:81] Initial profiling/warmup run took 0.19 s


Capturing CUDA graphs (FULL): 100%|██████████| 1/1 [00:00<00:00,  7.82it/s]


(EngineCore pid=88358) INFO 09-27 01:18:25 [model_runner.py:1066] Graph capturing finished in 1 secs, took 0.04 GiB
(EngineCore pid=88358) INFO 09-27 01:18:25 [gpu_worker.py:640] Available KV cache memory: 4.5 GiB
(EngineCore pid=88358) INFO 09-27 01:18:25 [gpu_worker.py:655] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9500 is equivalent to --gpu-memory-utilization=0.9484 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9516. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=88358) INFO 09-27 01:18:25 [kv_cache_utils.py:2395] GPU KV cache size: 84,288 tokens, Maximum concurrency for 512 tokens per request: 164.62x
(EngineCore pid=88358) INFO 09-27 01:18:25 [kernel_warmup.py:172] JIT kernel warmup starting.
(EngineCore pid=88358) INFO 09-27 01:18:25 [kernel_warmup.py:185] JIT kernel warmup finished in 0.00s.
(EngineCore p

(EngineCore pid=88358) Process EngineCore:
(EngineCore pid=88358) Traceback (most recent call last):
(EngineCore pid=88358)   File "/opt/conda/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=88358)     self.run()
(EngineCore pid=88358)   File "/opt/conda/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=88358)     self._target(*self._args, **self._kwargs)
(EngineCore pid=88358)   File "/home/sagemaker-user/decoding/.venv-vllm/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1370, in run_engine_core
(EngineCore pid=88358)     raise e
(EngineCore pid=88358)   File "/home/sagemaker-user/decoding/.venv-vllm/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1328, in run_engine_core
(EngineCore pid=88358)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=88358)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=88358)   File "/home/sagem

CalledProcessError: Command '['.venv-vllm/bin/python', 'vllm_benchmark.py']' returned non-zero exit status 1.